# Compute Pool Sizing Guide for ML Workloads on Snowflake

This notebook provides a comprehensive guide to **t-shirt sizing** your Snowflake Compute Pool based on your ML use-case. It covers:

1. **Instance Family Reference** — Available CPU, High-Memory, and GPU options
2. **Use-Case Decision Matrix** — Match your workload to the right compute tier
3. **Sizing Formulas & Rules of Thumb** — Calculate nodes, memory, and workers
4. **Interactive Sizing Tool** — Input your workload parameters, get a recommendation
5. **Best Practices** — Cost optimization, auto-suspend, monitoring

---

## Key Concepts

| Term | Definition |
|------|------------|
| **Compute Pool** | A set of VM nodes managed by Snowflake for running containers (ML Jobs, Inference Services, Notebooks) |
| **Instance Family** | The hardware spec (vCPUs, RAM, GPUs) of each node in the pool |
| **MIN_NODES / MAX_NODES** | Elasticity bounds — pool scales between these limits |
| **AUTO_SUSPEND_SECS** | Idle timeout before nodes are released (saves cost) |
| **Workers** | Parallel execution units within a node (configurable via `num_cpus_per_worker`) |

## Instance Family Reference

### Current Generation (Recommended for new workloads)

#### General Compute (x86)
| Instance Family | vCPUs | Memory (GiB) | Storage (GB) | Best For |
|-----------------|-------|--------------|--------------|----------|
| `GEN_X64_G2_2` | 1 | 6 | 100 | Lightweight scripts, small model inference |
| `GEN_X64_G2_4` | 3 | 13 | 100 | Small sklearn models, feature engineering |
| `GEN_X64_G2_8` | 6 | 28 | 100 | Medium ML training, XGBoost, data preprocessing |
| `GEN_X64_G2_16` | 14 | 58 | 100 | Large tabular models, multi-threaded training |
| `GEN_X64_G2_32` | 28 | 116 | 100 | Large-scale CPU training, many-model training |

#### High Memory
| Instance Family | vCPUs | Memory (GiB) | Storage (GB) | Best For |
|-----------------|-------|--------------|--------------|----------|
| `MEM_X64_G2_8` | 6 | 58 | 100 | Large DataFrames, in-memory feature stores |
| `MEM_X64_G2_32` | 28 | 240 | 100 | Very large datasets, memory-intensive models |
| `MEM_X64_G2_64` | 60 | 492 | 100 | Massive in-memory workloads |
| `MEM_X64_G2_192` | 188 | 1436 | 100 | Extreme memory requirements |

#### GPU Accelerated (AWS)
| Instance Family | vCPUs | Memory (GiB) | GPUs | GPU Memory | Best For |
|-----------------|-------|--------------|------|------------|----------|
| `GPU_NV_S` | 6 | 28 | 1x A10G | 24 GB | Small model fine-tuning, single-GPU inference |
| `GPU_NV_M` | 44 | 178 | 4x A10G | 96 GB total | Multi-GPU training, medium LLM inference |
| `GPU_NV_L` | 92 | 1112 | 8x A100 | 320 GB total | Large-scale distributed training, LLM serving |
| `GPU_L40S_G1_8` | 6 | 58 | 1x L40S | 48 GB | GenAI inference, fine-tuning |
| `GPU_L40S_G1_48` | 44 | 368 | 4x L40S | 192 GB total | Multi-GPU GenAI workloads |
| `GPU_R6K_G1_8` | 6 | 58 | 1x RTX PRO 6000 | 96 GB | Large model inference, data-intensive AI |

### Previous Generation (Still available, lower cost)
| Instance Family | vCPUs | Memory (GiB) | Best For |
|-----------------|-------|--------------|----------|
| `CPU_X64_XS` | 1 | 6 | Minimal workloads, testing |
| `CPU_X64_S` | 3 | 13 | Small batch jobs |
| `CPU_X64_M` | 6 | 28 | Standard ML training |
| `CPU_X64_L` | 28 | 116 | Large CPU workloads |
| `HIGHMEM_X64_S` | 6 | 58 | Memory-bound workloads |
| `HIGHMEM_X64_M` | 28 | 240 | Large data processing |

## ML Use-Case Decision Matrix

Use this matrix to quickly identify the right compute tier for your workload:

| ML Use-Case | Recommended Tier | Instance Family | Nodes | Rationale |
|-------------|-----------------|-----------------|-------|-----------|
| **Small sklearn model** (logistic reg, random forest <1M rows) | CPU Small | `GEN_X64_G2_4` or `CPU_X64_S` | 1-3 | Fits in memory, fast training |
| **XGBoost / LightGBM** (1M-100M rows) | CPU Medium | `GEN_X64_G2_8` or `CPU_X64_M` | 1-5 | Multi-threaded, moderate memory |
| **Distributed XGBoost** (>100M rows) | CPU Large | `GEN_X64_G2_32` or `CPU_X64_L` | 3-10 | Ray distributes across nodes |
| **Many-Model Training** (model per partition) | CPU Small/Medium | `GEN_X64_G2_4` to `GEN_X64_G2_8` | 5-50 | Embarrassingly parallel |
| **Deep Learning** (PyTorch/TensorFlow) | GPU | `GPU_NV_S` to `GPU_NV_L` | 1-8 | GPU acceleration essential |
| **LLM Fine-Tuning** | GPU Large | `GPU_NV_M` or `GPU_L40S_G1_48` | 2-4 | Multi-GPU, high VRAM needed |
| **LLM Inference Serving** | GPU | `GPU_NV_S` or `GPU_L40S_G1_8` | 1-4 | Low latency, model fits in VRAM |
| **Hyperparameter Tuning** | CPU/GPU (match base) | Same as base training | 5-20 | Each trial = 1 worker |
| **Feature Engineering** (large joins/transforms) | High Memory | `MEM_X64_G2_8` to `MEM_X64_G2_32` | 1-5 | Memory-bound operations |
| **Batch Inference** (bulk scoring) | CPU Small | `GEN_X64_G2_4` or `CPU_X64_S` | 1-10 | I/O bound, low compute per row |
| **Real-Time Inference** (REST endpoint) | CPU/GPU (match model) | `GEN_X64_G2_4` or `GPU_NV_S` | 1-3 | Low latency, steady-state |

---

### Key Decision Factors

1. **Dataset Size** → Determines memory requirements
2. **Model Complexity** → Determines CPU/GPU needs  
3. **Parallelism Strategy** → Determines node count
4. **Latency Requirements** → Determines instance tier
5. **Cost Sensitivity** → Determines MIN_NODES and AUTO_SUSPEND

## Sizing Formulas & Rules of Thumb

### Memory Sizing

```
Required Memory per Node = (Dataset Size in Memory) × Overhead Multiplier

Overhead Multipliers:
  - Pandas operations: 3-5x raw data size
  - XGBoost training: 2-3x raw data size  
  - Deep Learning: model_params × 4 bytes (fp32) × 3 (weights + gradients + optimizer)
  - Spark/Ray shuffle: 2x partition size
```

### Node Count Estimation

```
For Distributed Training (Estimators):
  nodes = ceil(total_data_size / memory_per_node) + 1 (coordinator overhead)

For Many-Model Training (MMT/DPF):
  workers_per_node = node_vCPUs / cpus_per_worker
  ideal_nodes = ceil(num_partitions / workers_per_node)
  practical_nodes = min(ideal_nodes, 50)  # 50-node recommended max

For Hyperparameter Tuning:
  nodes = ceil(max_concurrent_trials / workers_per_node)

For Inference Services:
  nodes = ceil(target_QPS / QPS_per_node)
```

### Workers Per Node

| Scenario | CPUs per Worker | Reasoning |
|----------|----------------|----------|
| Few long-running tasks (minutes+) | Default (None) = 1 worker/node | Full resources per task |
| Many short tasks (seconds each) | 1 | Pack workers, reduce node wait time |
| Multi-threaded training per task | 4-8 | Match to thread count |
| OOM errors | Increase CPUs/worker | Fewer workers = more memory each |

### Cost Rule of Thumb

```
Total Cost ∝ instance_credits_per_hour × max_nodes × active_hours

Optimize by:
1. Setting MIN_NODES = 1 (scale from zero when idle)
2. Using AUTO_SUSPEND_SECS = 300-3600 (5-60 min idle timeout)
3. Right-sizing MAX_NODES (don't over-provision)
4. Choosing the smallest instance family that avoids OOM
```

In [ ]:
%%sql -r available_instances
SHOW COMPUTE POOL INSTANCE FAMILIES IN ACCOUNT;

## Best Practices & Cost Optimization

### Compute Pool Configuration Best Practices

| Practice | Recommendation | Why |
|----------|---------------|-----|
| **Start small, scale up** | Begin with `MIN_NODES=1`, increase `MAX_NODES` as needed | Avoid paying for idle capacity |
| **Use AUTO_SUSPEND** | 300s for dev/test, 3600s for production | Nodes cost credits even when idle |
| **Set AUTO_RESUME=TRUE** | Always enable | Pool restarts automatically on demand |
| **Separate pools by workload** | Training pool vs. Inference pool | Different SLA and scaling needs |
| **Monitor before scaling** | Check actual utilization before adding nodes | Over-provisioning is the #1 cost waste |

### Instance Family Selection Rules

1. **When in doubt, start with `GEN_X64_G2_8`** (6 vCPU, 28 GiB) — it handles most tabular ML workloads
2. **OOM errors?** → Move to High Memory (`MEM_X64_G2_*`) or increase `num_cpus_per_worker`
3. **Training too slow?** → Add more nodes (horizontal) before upgrading instance size (vertical)
4. **GPU selection:** Match GPU VRAM to your model size:
   - Model < 7B params → 1x A10G (24 GB) or 1x L40S (48 GB)
   - Model 7B-13B params → 1x L40S (48 GB) or 1x RTX PRO 6000 (96 GB)
   - Model 13B-70B params → 4x A10G or 4x L40S or multi-node
   - Model > 70B params → 8x A100 (320 GB) or multi-node GPU

### Node Count Guidelines

| Workload Pattern | MIN_NODES | MAX_NODES | AUTO_SUSPEND |
|-----------------|-----------|-----------|-------------|
| Dev/Experimentation | 1 | 1-3 | 300s |
| Scheduled Training | 1 | 5-20 | 600s |
| Many-Model (100s of partitions) | 1 | 10-50 | 1800s |
| Production Inference (REST) | 1-2 | 3-5 | Never (or 7200s) |
| Batch Scoring (scheduled) | 1 | 5-10 | 300s |
| HPO / Tuning | 1 | 10-20 | 600s |

### Common Pitfalls to Avoid

1. **Over-provisioning MAX_NODES** — You pay for nodes once they spin up, even if your job finishes quickly
2. **Forgetting AUTO_SUSPEND** — Without it, nodes stay active indefinitely
3. **Wrong instance for GPU workloads** — CPU instances for deep learning waste time and money
4. **Too many nodes for small workloads** — Node provisioning takes 30-120s; for jobs < 5 min, fewer pre-warmed nodes > many cold nodes
5. **Not checking availability** — Run `SHOW COMPUTE POOL INSTANCE FAMILIES` to see what's in your region

In [ ]:
%%sql -r existing_pools
SHOW COMPUTE POOLS;